In [1]:
!pip install pandas openpyxl mysql-connector-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 75.0 MB/s eta 0:00:00


In [2]:
import pandas as pd
import mysql.connector
import getpass


In [3]:
df = pd.read_excel("Rhode_Synthetic_Sales_Dataset_10000.xlsx")
df.head()

,Order_ID,Order_Date,Product,Category,Unit_Price,Quantity,Sales,Customer_Rating,Review_Count,Country,Payment_Method,Discount_%,Revenue,Profit,Season,Marketing_Channel,Returned
0,ORD100002,2026-05-28,Pineapple Refresh Cleanser,Cleanser,28,1,26.6,4.2,373,USA,PayPal,5,26.6,14.08,Spring,Instagram,Yes
1,ORD100003,2024-08-31,Glazing Milk,Skincare,30,5,127.5,3.8,3620,UK,Apple Pay,15,127.5,44.67,Winter,Email,No
2,ORD100004,2024-05-03,Peptide Lip Treatment,Lip Care,18,4,57.6,4.4,4817,UK,Google Pay,20,57.6,30.54,Winter,Email,Yes
3,ORD100005,2023-03-27,Pocket Blush,Makeup,24,4,96.0,4.4,1837,Canada,Card,0,96.0,45.22,Spring,Email,Yes
4,ORD100006,2026-05-18,Pocket Blush,Makeup,24,2,45.6,4.7,3553,India,Apple Pay,5,45.6,24.47,Summer,Organic,No


In [4]:
df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")

In [7]:
password = getpass.getpass("Enter Aiven Password: ")

conn = mysql.connector.connect(
    host="mysql-76d5140-letmethink2006-2409.d.aivencloud.com",
    port=20810,
    user="avnadmin",
    password=password,
    database="defaultdb",
    ssl_disabled=False
)

Enter Aiven Password: ··········


In [ ]:
# Create table
cursor.execute("""
CREATE TABLE IF NOT EXISTS rhode_sales (
    id CHAR(36) NOT NULL PRIMARY KEY DEFAULT (UUID()),
    Order_ID VARCHAR(20) NOT NULL,
    Order_Date DATE,
    Product VARCHAR(100),
    Category VARCHAR(50),
    Unit_Price DECIMAL(10,2),
    Quantity INT,
    Sales DECIMAL(10,2),
    Customer_Rating DECIMAL(3,1),
    Review_Count INT,
    Country VARCHAR(50),
    Payment_Method VARCHAR(50),
    Discount_Percent INT,
    Revenue DECIMAL(10,2),
    Profit DECIMAL(10,2),
    Season VARCHAR(20),
    Marketing_Channel VARCHAR(50),
    Returned VARCHAR(5)
)
""")

In [ ]:
insert_sql = """
INSERT INTO rhode_sales (
    Order_ID,
    Order_Date,
    Product,
    Category,
    Unit_Price,
    Quantity,
    Sales,
    Customer_Rating,
    Review_Count,
    Country,
    Payment_Method,
    Discount_Percent,
    Revenue,
    Profit,
    Season,
    Marketing_Channel,
    Returned
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

In [ ]:
data = []

for _, row in df.iterrows():
    data.append((
        row["Order_ID"],
        row["Order_Date"].date() if pd.notna(row["Order_Date"]) else None,
        row["Product"],
        row["Category"],
        float(row["Unit_Price"]),
        int(row["Quantity"]),
        float(row["Sales"]),
        float(row["Customer_Rating"]),
        int(row["Review_Count"]),
        row["Country"],
        row["Payment_Method"],
        int(row["Discount_%"]),
        float(row["Revenue"]),
        float(row["Profit"]),
        row["Season"],
        row["Marketing_Channel"],
        row["Returned"]
    ))

cursor.executemany(insert_sql, data)
conn.commit()

print(f"{cursor.rowcount} rows inserted.")

0 rows inserted.
